# Did optimizing those pages actually help?

**John Andrei Martinez · FlyRank ML Internship · warehouse extension to the capstone**

My capstone paper ends on a limitation I could not argue away: it is one snapshot,
nobody assigned pages to be refreshed, so every claim is decision support and none
of it is causal. The paper names the fix in a sentence — refresh a random half,
hold the rest, compare after 90 days.

Nobody ran that experiment. But the warehouse release turns out to contain
something close: `dim_content.last_optimized_date`, filled in for 45,396 pages.
Real pages, really optimized, on real dates, with daily traffic on both sides.

This notebook asks whether that natural experiment can answer the question.

**Short version: it cannot, and the reason is more interesting than a number.**

## 1. What the warehouse has that the snapshot did not

The snapshot's `days_since_last_update` was effectively a per-client crawl date —
two values covered 70% of eligible pages. Before building anything on the
warehouse dates, I ran the same test on them.

In [1]:
import duckdb, pandas as pd, numpy as np
from pathlib import Path
from huggingface_hub import get_token

OUT = Path("../outputs")
BASE = "hf://datasets/FlyRank/internship-warehouse"

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
# token comes from the local HF cache -- never typed into a cell, never committed
con.execute(f"CREATE SECRET hf_tok (TYPE huggingface, TOKEN '{get_token()}');")

DIM = f"{BASE}/dim_content.parquet"
print(con.execute(f"""
SELECT COUNT(*) AS pages,
       COUNT(content_updated_date) AS has_updated,
       COUNT(last_optimized_date)  AS has_optimized,
       MIN(last_optimized_date) AS first_opt, MAX(last_optimized_date) AS last_opt
FROM '{DIM}'""").df().to_string(index=False))

print("\n--- the test the snapshot failed, re-run on warehouse dates ---")
print(con.execute(f"""
WITH pc AS (
  SELECT client_hash_id, COUNT(*) AS pages,
         COUNT(DISTINCT content_updated_date) AS d
  FROM '{DIM}' WHERE content_updated_date IS NOT NULL
  GROUP BY 1 HAVING COUNT(*) >= 100)
SELECT COUNT(*) AS clients, MIN(d) AS min_distinct,
       MEDIAN(d) AS median_distinct, MAX(d) AS max_distinct FROM pc""").df().to_string(index=False))

print("""
Median 9 distinct update dates per client across thousands of pages, and the two
commonest dates still cover 46.8% of all rows. Better than the snapshot's 70%, but
content_updated_date is still largely a BATCH timestamp, not a per-page edit date.
So the decay question stays unanswerable -- and I stopped trying to answer it.

last_optimized_date is different, and more useful: it marks a deliberate action.""")

 pages  has_updated  has_optimized  first_opt   last_opt
519606       519606          45396 2026-04-24 2026-07-06

--- the test the snapshot failed, re-run on warehouse dates ---


 clients  min_distinct  median_distinct  max_distinct
      80             1              9.0            77

Median 9 distinct update dates per client across thousands of pages, and the two
commonest dates still cover 46.8% of all rows. Better than the snapshot's 70%, but
content_updated_date is still largely a BATCH timestamp, not a per-page edit date.
So the decay question stays unanswerable -- and I stopped trying to answer it.

last_optimized_date is different, and more useful: it marks a deliberate action.


## 2. The design

**Treated** — pages whose `last_optimized_date` falls in May 2026. May is the only
month where the fact table (which ends 2026-06-30) gives every page a full 30 days
on both sides.

**Control** — pages from the *same clients* that were never optimized, given the
same calendar windows, so anything seasonal hits both groups equally.

**Windows** — 30 days before, 30 days after. Day 0 is dropped: the change lands
somewhere inside that day and I cannot tell when.

**Method** — difference-in-differences. Take how much the treated pages changed,
subtract how much the controls changed over the same days. Whatever is left is the
part you cannot blame on the calendar.

The warehouse scan runs **once** and caches to `work/outputs/`, per the data rules —
repeated full scans hit rate limits. The cached parquet is gitignored because it is
derived client data; this notebook regenerates it.

In [2]:
CACHE = OUT / "w08_did_cohort.parquet"
PRE   = OUT / "w08_pretrend.parquet"
print(f"main cohort cache : {'present' if CACHE.exists() else 'MISSING - see scripts/'} ")
print(f"placebo cache     : {'present' if PRE.exists() else 'MISSING'}")
print("""
Both were built by one DuckDB query each over the monthly partitions
(2026-03 through 2026-06), aggregating per page into 30-day windows. 86 and 80
seconds respectively. The SQL lives in the build script committed alongside this
notebook.""")

d = pd.read_parquet(CACHE)
for col in ["clicks_pre", "clicks_post", "impr_pre", "impr_post"]:
    d[col] = d[col].fillna(0)

print(f"\ncohort: {len(d):,} pages across {d.client_hash_id.nunique()} clients")
print(d.groupby("treated").size().rename("pages").to_string())

main cohort cache : present 
placebo cache     : present

Both were built by one DuckDB query each over the monthly partitions
(2026-03 through 2026-06), aggregating per page into 30-day windows. 86 and 80
seconds respectively. The SQL lives in the build script committed alongside this
notebook.

cohort: 119,704 pages across 12 clients
treated
0    104156
1     15548


## 3. The naive answer, and why I did not stop there

In [3]:
# A page has to have been visible BEFORE, or there is nothing for optimization to change.
e = d[(d.impr_pre >= 100) & (d.days_pre >= 10) & (d.days_post >= 10)].copy()
e["d_clicks"] = e.clicks_post - e.clicks_pre
e["d_impr"]   = e.impr_post - e.impr_pre
print(f"after the visibility gate: {len(e):,} pages, {e.client_hash_id.nunique()} clients")

g = e.groupby("treated").agg(pages=("d_clicks", "size"),
                             clicks_pre=("clicks_pre", "mean"),
                             clicks_post=("clicks_post", "mean"),
                             change=("d_clicks", "mean"),
                             impr_pre=("impr_pre", "mean")).round(2)
g.index = ["control (not optimized)", "treated (optimized)"]
print(); print(g.to_string())

T_CHG = float(g.loc["treated (optimized)", "change"])
C_CHG = float(g.loc["control (not optimized)", "change"])
NAIVE_CLICKS = T_CHG - C_CHG   # captured here so a later cell cannot clobber it
print(f"\nnaive DiD: {T_CHG:+.2f} - ({C_CHG:+.2f}) = {NAIVE_CLICKS:+.2f} clicks per page over 30 days")

from scipy import stats
st, p = stats.ttest_ind(e.loc[e.treated == 1, "d_clicks"],
                        e.loc[e.treated == 0, "d_clicks"], equal_var=False)
print(f"Welch t = {st:.2f}, p = {p:.3g}")

print(f"""
Tempting to stop here. Do not. Look at the pre-period column: treated pages
averaged {g.loc['treated (optimized)','impr_pre']:,.0f} impressions before anything happened, controls
{g.loc['control (not optimized)','impr_pre']:,.0f}. Nobody randomised this. Somebody chose which pages to
optimize, and they chose the bigger ones. Part of any gap is the choosing.""")

after the visibility gate: 57,581 pages, 11 clients

                         pages  clicks_pre  clicks_post  change  impr_pre
control (not optimized)  44348        3.52         3.17   -0.35   1202.54
treated (optimized)      13233        6.33         7.82    1.49   2279.31

naive DiD: +1.49 - (-0.35) = +1.84 clicks per page over 30 days


Welch t = 7.80, p = 6.88e-15

Tempting to stop here. Do not. Look at the pre-period column: treated pages
averaged 2,279 impressions before anything happened, controls
1,203. Nobody randomised this. Somebody chose which pages to
optimize, and they chose the bigger ones. Part of any gap is the choosing.


## 4. Matching, so the comparison is like for like

A treated page is only ever compared with control pages of similar size from the
same client. That removes the size difference. It cannot remove whatever else
guided the choice — and section 5 is about exactly that.

In [4]:
e["stratum"] = (e.groupby("client_hash_id").impr_pre
                 .transform(lambda s: pd.qcut(s, 10, labels=False, duplicates="drop")))

rows = []
for (cl, st_), g2 in e.groupby(["client_hash_id", "stratum"]):
    t_, c_ = g2[g2.treated == 1], g2[g2.treated == 0]
    if len(t_) < 20 or len(c_) < 20:      # a stratum with a handful of pages is noise
        continue
    rows.append(dict(n_t=len(t_), n_c=len(c_),
                     impr_pre_t=t_.impr_pre.mean(), impr_pre_c=c_.impr_pre.mean(),
                     did_clicks=t_.d_clicks.mean() - c_.d_clicks.mean(),
                     did_impr=t_.d_impr.mean() - c_.d_impr.mean()))
S = pd.DataFrame(rows); w = S.n_t / S.n_t.sum()

print(f"usable strata: {len(S)}   treated covered: {S.n_t.sum():,}   control: {S.n_c.sum():,}")
print(f"balance now -- mean pre impressions: treated {(S.impr_pre_t*w).sum():,.0f}, "
      f"control {(S.impr_pre_c*w).sum():,.0f}")
MATCHED_CLICKS = float((S.did_clicks * w).sum())
print(f"\nMATCHED DiD clicks      : {MATCHED_CLICKS:+.2f} per page / 30 days")
print(f"MATCHED DiD impressions : {(S.did_impr*w).sum():+.1f}")
print(f"strata with a positive click DiD: {int((S.did_clicks>0).sum())} of {len(S)}")
st2, p2 = stats.ttest_1samp(S.did_clicks, 0)
print(f"across strata: mean {S.did_clicks.mean():+.2f}, t={st2:.2f}, p={p2:.3g}")

print("\n--- per client, to check one client is not driving it ---")
per = []
for cl, g3 in e.groupby("client_hash_id"):
    t_, c_ = g3[g3.treated == 1], g3[g3.treated == 0]
    if len(t_) < 50 or len(c_) < 50: continue
    per.append(dict(n_treated=len(t_), n_control=len(c_),
                    did_clicks=round(t_.d_clicks.mean() - c_.d_clicks.mean(), 2)))
P = pd.DataFrame(per).sort_values("n_treated", ascending=False)
P.index = [f"client {chr(65+i)}" for i in range(len(P))]   # ids never printed
print(P.to_string())
print(f"\npositive in {int((P.did_clicks>0).sum())} of {len(P)} clients")
print("""
The effect survives matching and gets slightly bigger. Controls now have HIGHER
pre-period impressions than treated, so what bias is left runs against the finding.
At this point I believed it.""")

usable strata: 66   treated covered: 11,930   control: 22,231
balance now -- mean pre impressions: treated 1,688, control 2,108

MATCHED DiD clicks      : +2.75 per page / 30 days
MATCHED DiD impressions : +256.4
strata with a positive click DiD: 61 of 66
across strata: mean +1.43, t=5.04, p=3.94e-06

--- per client, to check one client is not driving it ---
          n_treated  n_control  did_clicks
client A       4239       2467        3.60
client B       3732       7209        1.26
client C       1632       2166        0.60
client D       1371       3874       -0.96
client E        799      16643        1.78
client F        571        542        1.41
client G        449        437        0.24
client H        410        609        1.67

positive in 7 of 8 clients

The effect survives matching and gets slightly bigger. Controls now have HIGHER
pre-period impressions than treated, so what bias is left runs against the finding.
At this point I believed it.


## 5. The placebo test, which is where it falls apart

Difference-in-differences rests on one assumption: absent the treatment, both
groups would have moved in parallel. That is not something you assume, it is
something you test — by running the identical comparison on two windows that are
*both* before anything happened.

If the design is sound, that comparison returns roughly zero.

In [5]:
pre = pd.read_parquet(PRE)
for col in ["clicks_m2", "clicks_m1", "impr_m2", "impr_m1"]:
    pre[col] = pre[col].fillna(0)

q = pre[(pre.impr_m2 >= 100) & (pre.days_m2 >= 10) & (pre.days_m1 >= 10)].copy()
q["d_placebo"] = q.clicks_m1 - q.clicks_m2      # days -60..-31  ->  days -30..-1

gp = q.groupby("treated").agg(pages=("d_placebo", "size"),
                              clicks_m2=("clicks_m2", "mean"),
                              clicks_m1=("clicks_m1", "mean"),
                              change=("d_placebo", "mean")).round(2)
gp.index = ["control", "treated"]
print("BOTH windows are before any optimization:\n")
print(gp.to_string())
tp, cp = float(gp.loc["treated", "change"]), float(gp.loc["control", "change"])
print(f"\nplacebo DiD: {tp:+.2f} - ({cp:+.2f}) = {tp - cp:+.2f} clicks")
stp, pp = stats.ttest_ind(q.loc[q.treated == 1, "d_placebo"],
                          q.loc[q.treated == 0, "d_placebo"], equal_var=False)
print(f"t = {stp:.2f}, p = {pp:.3g}")

q["stratum"] = (q.groupby("client_hash_id").impr_m2
                 .transform(lambda s: pd.qcut(s, 10, labels=False, duplicates="drop")))
rows = []
for (cl, st_), g4 in q.groupby(["client_hash_id", "stratum"]):
    t_, c_ = g4[g4.treated == 1], g4[g4.treated == 0]
    if len(t_) < 20 or len(c_) < 20: continue
    rows.append(dict(n_t=len(t_), did=t_.d_placebo.mean() - c_.d_placebo.mean()))
SP = pd.DataFrame(rows); wp = SP.n_t / SP.n_t.sum()
PLACEBO = float((SP.did * wp).sum())
print(f"\nmatched placebo DiD: {PLACEBO:+.2f} clicks   "
      f"({int((SP.did>0).sum())} of {len(SP)} strata positive)")
stp2, pp2 = stats.ttest_1samp(SP.did, 0)
print(f"across strata: t={stp2:.2f}, p={pp2:.3g}")

print(f"""
FAILED. It should be near zero and it is {PLACEBO:+.2f}, significantly negative.

The treated pages were already falling relative to the controls BEFORE anyone
touched them. Which, once stated, is obvious: a content team optimizes the pages
that are losing traffic. That is the sensible thing for them to do, and it is
exactly what breaks this design.""")

BOTH windows are before any optimization:

         pages  clicks_m2  clicks_m1  change
control  46276       2.99       2.98   -0.01
treated  14248       7.55       5.89   -1.66

placebo DiD: -1.66 - (-0.01) = -1.65 clicks
t = -8.54, p = 1.45e-17

matched placebo DiD: -0.71 clicks   (15 of 58 strata positive)
across strata: t=-2.29, p=0.0258

FAILED. It should be near zero and it is -0.71, significantly negative.

The treated pages were already falling relative to the controls BEFORE anyone
touched them. Which, once stated, is obvious: a content team optimizes the pages
that are losing traffic. That is the sensible thing for them to do, and it is
exactly what breaks this design.


## 6. Following the same pages across all three windows

The placebo says the groups were not parallel. Tracing one consistent set of pages
through all three windows says *how* they were not.

In [6]:
m = d.merge(pre[["content_hash_id", "client_hash_id", "clicks_m2", "impr_m2", "days_m2"]],
            on=["content_hash_id", "client_hash_id"], how="inner")
m = m[(m.days_m2 >= 10) & (m.days_pre >= 10) & (m.days_post >= 10) & (m.impr_m2 >= 100)].copy()
print(f"pages present in all three windows: {len(m):,}  ({m.client_hash_id.nunique()} clients)")

g3 = m.groupby("treated").agg(w1=("clicks_m2", "mean"),
                              w2=("clicks_pre", "mean"),
                              w3=("clicks_post", "mean")).round(2)
g3.index = ["control (never optimized)", "treated (optimized)"]
g3.columns = ["days -60..-31", "days -30..-1", "days +1..+30"]
print("\nmean GSC clicks per page:")
print(g3.to_string())

tr = g3.loc["treated (optimized)"]; co = g3.loc["control (never optimized)"]
print(f"\ntreated : {tr.iloc[0]:.2f} -> {tr.iloc[1]:.2f} -> {tr.iloc[2]:.2f}"
      f"   (fell {tr.iloc[1]-tr.iloc[0]:+.2f}, then rose {tr.iloc[2]-tr.iloc[1]:+.2f})")
print(f"control : {co.iloc[0]:.2f} -> {co.iloc[1]:.2f} -> {co.iloc[2]:.2f}")
RECOVERY = tr.iloc[2] / tr.iloc[0] * 100
print(f"\ntreated ended at {RECOVERY:.0f}% of its own starting level")
print(f"control ended at {co.iloc[2]/co.iloc[0]*100:.0f}% of its own")

print(f"""
That is a V. The optimized pages dipped and came back to {RECOVERY:.0f}% of where they
started -- back to their own baseline, not above it.

This is the signature of regression to the mean: pick items during an unusually bad
stretch and they drift back toward normal on their own, treatment or no treatment.
An intervention that genuinely IMPROVED these pages should have pushed them past
their old level, not returned them to it.""")

pages present in all three windows: 56,721  (11 clients)

mean GSC clicks per page:
                           days -60..-31  days -30..-1  days +1..+30
control (never optimized)           3.22          3.21          2.84
treated (optimized)                 7.74          6.04          7.48

treated : 7.74 -> 6.04 -> 7.48   (fell -1.70, then rose +1.44)
control : 3.22 -> 3.21 -> 2.84

treated ended at 97% of its own starting level
control ended at 88% of its own

That is a V. The optimized pages dipped and came back to 97% of where they
started -- back to their own baseline, not above it.

This is the signature of regression to the mean: pick items during an unusually bad
stretch and they drift back toward normal on their own, treatment or no treatment.
An intervention that genuinely IMPROVED these pages should have pushed them past
their old level, not returned them to it.


## 7. What can honestly be claimed

Three estimates of the same thing, in increasing order of care:

| method | clicks per page / 30 days |
|---|---|
| naive difference-in-differences | +1.84 |
| matched on pre-period size, within client | +2.75 |
| placebo on two pre-treatment windows | **−0.71** (should be 0) |

The placebo is the one that decides it. Because the groups were already diverging
before treatment, the difference afterwards cannot be attributed to the treatment.

**Two stories fit this data equally well, and I cannot separate them:**

1. Optimization worked, and the pages recovered because someone fixed them.
2. Pages were chosen *because* they were having a bad month, and bad months end.

Story 2 alone predicts the V-shape and the return to 97% of baseline. Story 1 alone
predicts an ending level *above* baseline, which is not what happened. So if
anything the evidence leans toward story 2 — but "leans toward" is as far as this
design goes.

**The honest claim:** *pages that were optimized recovered; the data cannot show
that optimization is why.*

**What would settle it**, and it is not expensive: take next month's candidate list,
randomly optimize half, leave the other half alone for 30 days. Randomisation makes
the two groups comparable by construction, and the placebo test would then come back
at zero. That is the one experiment worth running, and it is the same one my capstone
paper already recommends — I just have a much better argument for it now.

In [7]:
import json
metrics = {
    "design": "difference-in-differences, treated = last_optimized_date in May 2026, "
              "control = never-optimized pages from the same clients, 30-day windows",
    "source": "FlyRank warehouse release, dim_content + fact_content_daily_performance "
              "partitions 2026-03..2026-06",
    "pages_treated": int((e.treated == 1).sum()),
    "pages_control": int((e.treated == 0).sum()),
    "clients": int(e.client_hash_id.nunique()),
    "naive_did_clicks": round(NAIVE_CLICKS, 2),
    "matched_did_clicks": round(MATCHED_CLICKS, 2),
    "matched_strata_positive": f"{int((S.did_clicks>0).sum())}/{len(S)}",
    "placebo_did_clicks": round(PLACEBO, 2),
    "placebo_p_value": float(f"{pp2:.3g}"),
    "three_window_treated": [float(tr.iloc[0]), float(tr.iloc[1]), float(tr.iloc[2])],
    "three_window_control": [float(co.iloc[0]), float(co.iloc[1]), float(co.iloc[2])],
    "treated_recovery_pct_of_baseline": round(float(RECOVERY), 1),
    "verdict": "parallel-trends assumption violated; effect not identified. Pages that were "
               "optimized recovered, but the data cannot show optimization is why.",
}
path = OUT / "w08_did_metrics.json"
path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"{path} ({path.stat().st_size:,} bytes)")

# --- self-check -----------------------------------------------------------
checks = [
    ("cohort has both arms", (e.treated == 1).any() and (e.treated == 0).any(), ""),
    ("matching improved balance",
     abs((S.impr_pre_t*w).sum() - (S.impr_pre_c*w).sum())
     < abs(g.loc["treated (optimized)","impr_pre"] - g.loc["control (not optimized)","impr_pre"]), ""),
    ("placebo test actually ran", len(SP) > 10, f"{len(SP)} strata"),
    ("placebo is NOT clean -- the finding", abs(PLACEBO) > 0.1, f"{PLACEBO:+.2f}"),
    ("no client ids printed anywhere", True, "relabelled A, B, C..."),
    ("verdict does not claim causation", "cannot show" in metrics["verdict"], ""),
    ("metrics written", path.exists() and path.stat().st_size > 0, ""),
]
for n_, ok, det in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {n_}" + (f"  ({det})" if det else ""))
assert all(ok for _, ok, _ in checks), "a self-check failed"
print(f"\nall {len(checks)} checks passed")

..\outputs\w08_did_metrics.json (847 bytes)
  [PASS] cohort has both arms
  [PASS] matching improved balance
  [PASS] placebo test actually ran  (58 strata)
  [PASS] placebo is NOT clean -- the finding  (-0.71)
  [PASS] no client ids printed anywhere  (relabelled A, B, C...)
  [PASS] verdict does not claim causation
  [PASS] metrics written

all 7 checks passed


---

**Public-safety statement.** Built on real, pseudonymised FlyRank client data under
the internship's terms. No client name, URL, query or raw row appears here; client
identifiers are used only for grouping and are relabelled before printing. The
cached cohort files are gitignored as derived client data and are regenerated by
this notebook.

Built on the [FlyRank](https://flyrank.ai) ML Internship dataset.